# PPO: Proximal Policy Optimization

This notebook implements **PPO** (Schulman et al., 2017), the policy gradient algorithm
that powers RLHF (Reinforcement Learning from Human Feedback) in systems like ChatGPT.
PPO is the most widely used RL algorithm in practice due to its simplicity and stability.

We will:
1. Understand why policy gradients are an alternative to value-based methods (DQN)
2. Derive the policy gradient theorem and the PPO clipped objective
3. Implement an Actor-Critic network and rollout buffer from scratch
4. Understand Generalized Advantage Estimation (GAE)
5. Train PPO on CartPole-v1 and visualize the learning process
6. Compare PPO vs DQN

**Reference:** Schulman, J. et al. (2017). *Proximal Policy Optimization Algorithms.*
https://arxiv.org/abs/1707.06347

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
import random
import matplotlib.pyplot as plt
import pandas as pd
from src.utils.device import set_seed

try:
    import gymnasium as gym
    print(f"Gymnasium version: {gym.__version__}")
except ImportError:
    print("Gymnasium not installed. Run: pip install -e '.[rl]'")

device = "cpu"
set_seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

## 1. DQN vs Policy Gradients

In the DQN notebook, we learned a **value function** $Q(s, a)$ and derived the policy from it.
Policy gradient methods take a fundamentally different approach: learn the **policy directly**.

| | Value-Based (DQN) | Policy-Based (PPO) |
|---|---|---|
| Learns | Q-values → derive policy | Policy $\pi_\theta(a|s)$ directly |
| Action space | Discrete only | Discrete or continuous |
| Exploration | $\epsilon$-greedy (bolted on) | Built-in (stochastic policy) |
| Data reuse | Off-policy (replay buffer) | On-policy (fresh data each update) |
| Stability | Can diverge (moving targets) | More stable with PPO clipping |
| Sample efficiency | Higher (reuses data) | Lower (discards data after each update) |

## 2. The Policy Gradient Theorem

We parameterize the policy as $\pi_\theta(a|s)$ — a neural network that outputs action
probabilities given a state. The objective is to maximize expected return:

$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} [R(\tau)]$

The **policy gradient theorem** gives us the gradient:

$\nabla_\theta J(\theta) = \mathbb{E} [\nabla_\theta \log \pi_\theta(a|s) \cdot A(s, a)]$

where $A(s, a) = Q(s, a) - V(s)$ is the **advantage** — how much better action $a$ is
compared to the average action in state $s$.

**Intuition:** Increase the probability of actions with positive advantage (better than average),
decrease the probability of actions with negative advantage (worse than average).

## 3. Why PPO?

**Vanilla policy gradient** is unstable: a single large gradient step can destroy the policy,
and recovery is difficult because on-policy methods can't reuse old data.

**TRPO** (Trust Region Policy Optimization) constrains updates using a KL divergence constraint,
but requires expensive second-order optimization.

**PPO** achieves similar stability with a simple clipped objective — no second-order methods needed.

## 4. The PPO Clipped Objective

Define the probability ratio between the new and old policy:

$r_t(\theta) = \frac{\pi_\theta(a_t | s_t)}{\pi_{\theta_{\text{old}}}(a_t | s_t)}$

The PPO clipped surrogate objective:

$L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min \left( r_t(\theta) \hat{A}_t, \; \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon) \hat{A}_t \right) \right]$

where $\epsilon = 0.2$ (typically).

**How clipping works:**
- If $\hat{A}_t > 0$ (good action): $r_t$ is capped at $1 + \epsilon$. The policy can increase the action's probability, but not by more than 20%.
- If $\hat{A}_t < 0$ (bad action): $r_t$ is capped at $1 - \epsilon$. The policy can decrease the action's probability, but not by more than 20%.

This prevents catastrophically large policy updates.

## 5. Actor-Critic Architecture

PPO uses an **actor-critic** architecture — a single network with two heads:

```
State (4) → Shared(128 → ReLU → 128 → ReLU) → Actor head → action logits (2)
                                               → Critic head → state value (1)
```

- **Actor** outputs a probability distribution over actions (policy $\pi_\theta$)
- **Critic** estimates the state value $V(s)$ (used for computing advantages)
- **Shared trunk** extracts features useful for both tasks

In [ ]:
class ActorCritic(nn.Module):
    """Shared-trunk actor-critic network for discrete action spaces."""

    def __init__(self, state_size, action_size, hidden_size=128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
        )
        self.actor_head = nn.Linear(hidden_size, action_size)   # policy logits
        self.critic_head = nn.Linear(hidden_size, 1)            # state value

    def forward(self, x):
        features = self.shared(x)
        logits = self.actor_head(features)
        value = self.critic_head(features).squeeze(-1)
        return logits, value

    def act(self, state):
        """Sample action from policy, return (action, log_prob, value)."""
        if not isinstance(state, torch.Tensor):
            state = torch.tensor(state, dtype=torch.float32)
        state = state.unsqueeze(0)

        with torch.no_grad():
            logits, value = self.forward(state)
            dist = Categorical(logits=logits)
            action = dist.sample()
            log_prob = dist.log_prob(action)

        return action.item(), log_prob.item(), value.item()


# Demo
ac = ActorCritic(state_size=4, action_size=2)
state = torch.randn(1, 4)
logits, value = ac(state)
print(f"Logits: {logits.detach().numpy()} → probabilities: {torch.softmax(logits, -1).detach().numpy()}")
print(f"Value:  {value.item():.4f}")

action, log_prob, val = ac.act(np.random.randn(4))
print(f"\nSampled action: {action}, log_prob: {log_prob:.4f}, value: {val:.4f}")
print(f"Parameters: {sum(p.numel() for p in ac.parameters()):,}")

## 6. Generalized Advantage Estimation (GAE)

The advantage $A(s, a)$ tells us how much better an action is compared to average.
GAE (Schulman et al., 2016) computes it as an exponentially-weighted average of TD residuals:

$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$

$\hat{A}_t = \sum_{l=0}^{T-t} (\gamma \lambda)^l \delta_{t+l}$

The parameter $\lambda$ controls the bias-variance tradeoff:
- $\lambda = 0$: TD(0) advantage — low variance, high bias
- $\lambda = 1$: Monte Carlo advantage — high variance, low bias
- $\lambda = 0.95$: Standard choice — good balance

In [ ]:
class RolloutBuffer:
    """On-policy trajectory storage with GAE computation."""

    def __init__(self):
        self.states = []
        self.actions = []
        self.rewards = []
        self.values = []
        self.log_probs = []
        self.dones = []
        self.advantages = None
        self.returns = None

    def push(self, state, action, reward, value, log_prob, done):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.values.append(value)
        self.log_probs.append(log_prob)
        self.dones.append(done)

    def compute_returns_and_advantages(self, last_value, gamma=0.99, gae_lambda=0.95):
        """Compute GAE advantages and discounted returns."""
        n = len(self.rewards)
        advantages = np.zeros(n, dtype=np.float32)
        last_gae = 0.0

        for t in reversed(range(n)):
            if t == n - 1:
                next_value = last_value
            else:
                next_value = self.values[t + 1]

            next_non_terminal = 1.0 - self.dones[t]
            delta = self.rewards[t] + gamma * next_value * next_non_terminal - self.values[t]
            last_gae = delta + gamma * gae_lambda * next_non_terminal * last_gae
            advantages[t] = last_gae

        self.returns = advantages + np.array(self.values, dtype=np.float32)
        self.advantages = advantages

    def get_batches(self, batch_size, device="cpu"):
        """Yield shuffled mini-batches as tensors."""
        n = len(self.states)
        indices = np.random.permutation(n)

        states = torch.tensor(np.array(self.states), dtype=torch.float32, device=device)
        actions = torch.tensor(self.actions, dtype=torch.long, device=device)
        log_probs = torch.tensor(self.log_probs, dtype=torch.float32, device=device)
        returns = torch.tensor(self.returns, dtype=torch.float32, device=device)
        advantages = torch.tensor(self.advantages, dtype=torch.float32, device=device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            idx = indices[start:end]
            yield {
                "states": states[idx],
                "actions": actions[idx],
                "old_log_probs": log_probs[idx],
                "returns": returns[idx],
                "advantages": advantages[idx],
            }

    def clear(self):
        self.states.clear()
        self.actions.clear()
        self.rewards.clear()
        self.values.clear()
        self.log_probs.clear()
        self.dones.clear()
        self.advantages = None
        self.returns = None

    def __len__(self):
        return len(self.states)


# Demo: store a short trajectory and compute GAE
buf = RolloutBuffer()
for i in range(10):
    buf.push(state=np.random.randn(4), action=0, reward=1.0,
             value=0.5, log_prob=-0.7, done=False)

buf.compute_returns_and_advantages(last_value=0.5, gamma=0.99, gae_lambda=0.95)
print(f"Buffer size: {len(buf)}")
print(f"Advantages: {buf.advantages[:5]}")
print(f"Returns:    {buf.returns[:5]}")

## 7. On-Policy vs Off-Policy

PPO is **on-policy**: it must collect fresh data with the current policy for each update.
DQN is **off-policy**: it reuses old data from the replay buffer.

| | On-Policy (PPO) | Off-Policy (DQN) |
|---|---|---|
| Data collection | Fresh rollout each update | Store all transitions |
| Sample efficiency | Lower (discard after use) | Higher (reuse many times) |
| Stability | More stable | Can diverge |
| Buffer | Rollout buffer (cleared) | Replay buffer (persistent) |

PPO compensates for lower sample efficiency by doing **multiple gradient steps** (K epochs)
on each rollout, which is why the clipped objective is essential — it prevents the policy
from changing too much during these repeated updates.

## 8. The PPO Algorithm

```
Initialize actor-critic network θ
For each iteration:
    1. Collect T timesteps of experience using π_θ
    2. Compute GAE advantages Â_t and returns
    3. For K epochs:
        For each mini-batch:
            Compute ratio: r_t = π_θ(a|s) / π_θ_old(a|s)
            Clipped surrogate: L_clip = min(r_t Â_t, clip(r_t) Â_t)
            Value loss: L_vf = MSE(V(s), returns)
            Entropy bonus: H = -sum(π log π)
            Total loss: -L_clip + c1 * L_vf - c2 * H
            Gradient descent step
    4. Clear rollout buffer
```

In [ ]:
class PPOAgent:
    """PPO agent with clipped objective, value loss, and entropy bonus."""

    def __init__(self, state_size, action_size, hidden_size=128, lr=3e-4,
                 gamma=0.99, gae_lambda=0.95, clip_epsilon=0.2,
                 value_coeff=0.5, entropy_coeff=0.01, k_epochs=4,
                 batch_size=64, device="cpu"):
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.clip_epsilon = clip_epsilon
        self.value_coeff = value_coeff
        self.entropy_coeff = entropy_coeff
        self.k_epochs = k_epochs
        self.batch_size = batch_size
        self.device = device

        self.ac = ActorCritic(state_size, action_size, hidden_size).to(device)
        self.optimizer = optim.Adam(self.ac.parameters(), lr=lr)
        self.buffer = RolloutBuffer()

        # Logging
        self.policy_losses = []
        self.value_losses = []
        self.entropies = []

    def select_action(self, state):
        return self.ac.act(state)

    def update(self):
        """Run K epochs of PPO updates on the current rollout."""
        for _ in range(self.k_epochs):
            for batch in self.buffer.get_batches(self.batch_size, self.device):
                states = batch["states"]
                actions = batch["actions"]
                old_log_probs = batch["old_log_probs"]
                returns = batch["returns"]
                advantages = batch["advantages"]

                # Forward pass
                logits, values = self.ac(states)
                dist = Categorical(logits=logits)
                new_log_probs = dist.log_prob(actions)
                entropy = dist.entropy().mean()

                # Probability ratio
                ratio = torch.exp(new_log_probs - old_log_probs)

                # Clipped surrogate objective
                surr1 = ratio * advantages
                surr2 = torch.clamp(ratio, 1.0 - self.clip_epsilon,
                                    1.0 + self.clip_epsilon) * advantages
                policy_loss = -torch.min(surr1, surr2).mean()

                # Value loss
                value_loss = nn.functional.mse_loss(values, returns)

                # Total loss
                loss = policy_loss + self.value_coeff * value_loss - self.entropy_coeff * entropy

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.ac.parameters(), 0.5)
                self.optimizer.step()

                self.policy_losses.append(policy_loss.item())
                self.value_losses.append(value_loss.item())
                self.entropies.append(entropy.item())

        self.buffer.clear()


print("PPOAgent ready.")

In [ ]:
def train_ppo(agent, env, num_updates=200, rollout_length=2048, print_every=20):
    """Train a PPO agent and return episode rewards."""
    rewards_history = []
    state, _ = env.reset()
    episode_reward = 0

    for update in range(num_updates):
        # Collect rollout
        for _ in range(rollout_length):
            action, log_prob, value = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.buffer.push(state, action, reward, value, log_prob, float(done))
            state = next_state
            episode_reward += reward

            if done:
                rewards_history.append(episode_reward)
                episode_reward = 0
                state, _ = env.reset()

        # Compute advantages with last value estimate
        with torch.no_grad():
            _, _, last_value = agent.select_action(state)
        agent.buffer.compute_returns_and_advantages(
            last_value, agent.gamma, agent.gae_lambda
        )

        # PPO update
        agent.update()

        if (update + 1) % print_every == 0 and rewards_history:
            recent = rewards_history[-20:] if len(rewards_history) >= 20 else rewards_history
            print(f"Update {update+1:4d} | Episodes: {len(rewards_history):4d} | "
                  f"Avg reward (last 20): {np.mean(recent):6.1f}")

    return rewards_history


print("Training function ready.")

## 9. Loss Components

The total PPO loss combines three terms:

$L = -L^{CLIP} + c_1 \cdot L^{VF} - c_2 \cdot H[\pi]$

| Component | Purpose | Typical coefficient |
|---|---|---|
| $L^{CLIP}$ | Clipped policy loss | (maximized, so negated) |
| $L^{VF}$ | Value function MSE loss | $c_1 = 0.5$ |
| $H[\pi]$ | Entropy bonus (exploration) | $c_2 = 0.01$ |

The **entropy bonus** prevents premature convergence to a deterministic policy.
Higher entropy = more exploration.

## 10. Training

**Hyperparameters:**

| Parameter | Value | Description |
|---|---|---|
| Rollout length | 2048 | Timesteps per rollout |
| K epochs | 4 | Gradient steps per rollout |
| Clip $\epsilon$ | 0.2 | Ratio clipping range |
| $\gamma$ | 0.99 | Discount factor |
| GAE $\lambda$ | 0.95 | Advantage smoothing |
| Learning rate | $3 \times 10^{-4}$ | Adam optimizer |
| Batch size | 64 | Mini-batch size |
| Value coeff $c_1$ | 0.5 | Value loss weight |
| Entropy coeff $c_2$ | 0.01 | Entropy bonus weight |

In [ ]:
set_seed(42)
random.seed(42)
np.random.seed(42)

env = gym.make("CartPole-v1")

agent = PPOAgent(
    state_size=4,
    action_size=2,
    hidden_size=128,
    lr=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_epsilon=0.2,
    value_coeff=0.5,
    entropy_coeff=0.01,
    k_epochs=4,
    batch_size=64,
    device=device,
)

print("Training PPO on CartPole-v1...\n")
rewards = train_ppo(agent, env, num_updates=200, rollout_length=2048, print_every=20)

env.close()
print(f"\nTotal episodes: {len(rewards)}")
print(f"Final avg reward (last 50): {np.mean(rewards[-50:]):.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Episode Rewards
ax = axes[0]
ax.plot(rewards, alpha=0.3, color='tab:blue', label='Raw')
window = 50
rolling = pd.Series(rewards).rolling(window=window).mean()
ax.plot(rolling, color='tab:blue', linewidth=2, label=f'Rolling avg ({window})')
ax.axhline(y=475, color='tab:green', linestyle='--', alpha=0.5, label='Solved (475)')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Reward', fontsize=12)
ax.set_title('Episode Rewards', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 2: Policy Loss and Value Loss
ax = axes[1]
if agent.policy_losses:
    ax.plot(pd.Series(agent.policy_losses).rolling(50).mean(),
            color='tab:orange', linewidth=1, label='Policy loss')
    ax2 = ax.twinx()
    ax2.plot(pd.Series(agent.value_losses).rolling(50).mean(),
             color='tab:red', linewidth=1, label='Value loss')
    ax2.set_ylabel('Value Loss', fontsize=12, color='tab:red')
    ax2.legend(loc='upper right', fontsize=10)
ax.set_xlabel('Update Step', fontsize=12)
ax.set_ylabel('Policy Loss', fontsize=12, color='tab:orange')
ax.set_title('Training Losses', fontsize=14)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

# Plot 3: Entropy
ax = axes[2]
if agent.entropies:
    ax.plot(pd.Series(agent.entropies).rolling(50).mean(),
            color='tab:purple', linewidth=1)
ax.set_xlabel('Update Step', fontsize=12)
ax.set_ylabel('Entropy', fontsize=12)
ax.set_title('Policy Entropy (Exploration)', fontsize=14)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate the trained agent (greedy)
eval_env = gym.make("CartPole-v1")
eval_rewards = []

for ep in range(10):
    state, _ = eval_env.reset(seed=ep)
    episode_reward = 0
    done = False
    while not done:
        with torch.no_grad():
            state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
            logits, _ = agent.ac(state_t)
            action = logits.argmax().item()  # greedy
        state, reward, terminated, truncated, _ = eval_env.step(action)
        episode_reward += reward
        done = terminated or truncated
    eval_rewards.append(episode_reward)
    print(f"Eval episode {ep+1}: reward = {episode_reward}")

eval_env.close()
print(f"\nAvg eval reward: {np.mean(eval_rewards):.1f} (solved >= 475)")

## 11. PPO vs DQN on CartPole

| | DQN | PPO |
|---|---|---|
| Type | Value-based, off-policy | Policy gradient, on-policy |
| Network | Q-network (state → Q-values) | Actor-Critic (state → logits + value) |
| Data | Replay buffer (reused) | Rollout buffer (discarded) |
| Exploration | $\epsilon$-greedy decay | Stochastic policy + entropy bonus |
| Stability | Target network | Clipped objective |
| Actions | Discrete only | Discrete or continuous |
| Convergence | Faster on simple tasks | More stable on complex tasks |

## 12. Reusable Implementation

The ActorCritic and RolloutBuffer are available in `src/models/ppo.py`:

```python
from src.models.ppo import ActorCritic, RolloutBuffer
```

In [ ]:
from src.models.ppo import ActorCritic as AC, RolloutBuffer as RB

ac_test = AC(state_size=4, action_size=2)
logits, value = ac_test(torch.randn(1, 4))
print(f"ActorCritic output: logits={logits.shape}, value={value.shape}")

rb_test = RB()
rb_test.push(np.zeros(4), 0, 1.0, 0.5, -0.7, False)
print(f"RolloutBuffer length: {len(rb_test)}")
print("All src/ imports verified.")

## 13. Key Takeaways

1. **PPO learns the policy directly.** Instead of learning Q-values and deriving a policy, PPO parameterizes the policy as $\pi_\theta(a|s)$ and optimizes it with gradient ascent.

2. **The clipped objective prevents catastrophic updates.** By capping the probability ratio $r_t$ to $[1-\epsilon, 1+\epsilon]$, PPO ensures the policy doesn't change too much in a single update.

3. **GAE balances bias and variance in advantage estimation.** The $\lambda$ parameter smoothly interpolates between TD(0) (biased, low variance) and Monte Carlo (unbiased, high variance).

4. **PPO is on-policy but does K epochs per rollout.** It collects fresh data, then squeezes multiple gradient steps out of it. The clipping prevents overfitting to the rollout.

5. **Entropy bonus encourages exploration.** Without it, the policy can collapse to always choosing the same action. The coefficient $c_2$ controls the exploration-exploitation tradeoff.

6. **PPO is the most widely used RL algorithm in practice.** It powers RLHF in ChatGPT, robotics at scale (OpenAI, DeepMind), and game AI (Dota 2, StarCraft).

### Further Reading

- Schulman et al. (2017). *Proximal Policy Optimization Algorithms.* https://arxiv.org/abs/1707.06347
- Schulman et al. (2016). *High-Dimensional Continuous Control Using GAE.* https://arxiv.org/abs/1506.02438
- OpenAI Spinning Up — PPO: https://spinningup.openai.com/en/latest/algorithms/ppo.html
- Ouyang et al. (2022). *Training language models to follow instructions with RLHF.* https://arxiv.org/abs/2203.02155